# 01 · Ingesta y unión de las bases

**Objetivo.** Leer los microdatos del INEI y unirlos por `HHID`, verificando el match antes de seguir. El estrato solo existe en RECH0, así que esta unión es requisito para poder declarar el diseño muestral.

| | |
|---|---|
| **Entradas** | `CSALUD01_2025.sav`, `RECH0_2025.sav` |
| **Salidas** | DataFrame unido en memoria; el dataset final lo produce el notebook 03 |

> Proyecto: *Modelamiento epidemiológico de la depresión en la población peruana de 15 y
> más años (ENDES 2025)* · Epidemiología 2026-2 · UNMSM


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "src"))
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "scripts"))

import numpy as np
import pandas as pd
from depresion import config as cfg, io as dio, viz
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
viz.aplicar_tema()
print("Proyecto:", cfg.RAIZ.name)

## Lectura

Se lee con `apply_value_formats=False` **deliberadamente**: aplicar las etiquetas destruiría los 33 casos de violencia severa de `QS711`, cuya codificación real (3, 4) no coincide con el diccionario del INEI (8, 9).

In [ ]:
csalud, meta_cs = dio.leer_sav(cfg.ARCHIVO_CSALUD)
rech0, meta_r0 = dio.leer_sav(cfg.ARCHIVO_RECH0)
meta = meta_cs.combinar(meta_r0)
print(f"CSALUD01: {csalud.shape}")
print(f"RECH0   : {rech0.shape}")

## Verificación de la llave

Si el match cae por debajo del 99 %, el pipeline se detiene: no tiene sentido analizar una muestra que perdió parte de su geografía.

In [ ]:
izq = csalud[cfg.LLAVE].astype(str).str.strip()
der = rech0[cfg.LLAVE].astype(str).str.strip()
print(f"Ejemplo de HHID: {csalud[cfg.LLAVE].iloc[0]!r}")
print(f"Match: {izq.isin(set(der)).mean():.4%}")
print(f"Duplicados en RECH0: {der.duplicated().sum()}")

In [ ]:
from depresion import limpieza as lim
df = lim.unir_bases(csalud, rech0)
print(f"Tras la unión: {df.shape}")
df[[cfg.VAR_ESTRATO, cfg.VAR_DEPARTAMENTO, cfg.VAR_AREA, cfg.VAR_ALTITUD]].isna().sum()

## Conclusiones

- Match del **100,00 %** (33 569 de 33 569), sin duplicados. El `.str.strip()` no era
  estrictamente necesario —ambas claves traen el mismo relleno de 15 caracteres— pero se
  aplica por robustez.
- Tras la unión hay **0 faltantes** en estrato, departamento, área y altitud.
- `HV022` (estrato) vive **solo en RECH0**: sin esta unión no se puede declarar el diseño.
